# Prompt Engineering Best Practices: Text Summarization & Information Retrieval

[Article](https://todatabeyond.com/blog/prompt-engineering-best-practices-text-summarization-and-information-retrieval). [Series contents](./README.md).

Install `pip install openai==2.43.0 python-dotenv==1.2.2`. Set OPENAI_API_KEY in the environment or an uncommitted local .env. MODEL defaults to gpt-5.4-mini. Calls incur charges; never put a key in cells. Original code and output remain in the nonexecuting appendix. Examples keep the complete source input texts and loops; outputs can change between invocations.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion(prompt, model=MODEL):
    result = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        reasoning_effort="none", max_completion_tokens=2048)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def parse_json_object(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    raise ValueError("No JSON object found")
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
movie_review = """
Watched this movie over the weekend, and it turned out to be a delightful cinematic experience. The storyline was engaging, weaving together various plot elements seamlessly. The characters were well-developed, and their interactions felt genuine. The film had a perfect blend of drama, humor, and suspense, keeping me hooked from start to finish.

The visuals were stunning, with impressive cinematography that added depth to the narrative. The special effects were top-notch, enhancing the overall viewing experience. The soundtrack was also noteworthy, complementing the on-screen action and evoking the right emotions at the right moments.

Despite these strengths, there were a few drawbacks. Some scenes felt a bit rushed, leaving me wanting more in terms of character development. Additionally, the ending seemed a tad predictable, lacking the surprise factor that could have elevated the film to greater heights.

On the positive side, the movie was a bit of a hidden gem, exceeding my expectations. The pacing was well-managed, and the runtime felt just right. It left me with a sense of satisfaction, much like finding a surprise extra feature on a DVD.

In conclusion, this film is worth a watch for its compelling story, impressive visuals, and engaging characters. While it may not be a groundbreaking masterpiece, it certainly provides an enjoyable and entertaining escape into its cinematic world.

"""

In [ ]:
prompt = f"""
Your task is to generate a short summary of a movie  review from a streaming service. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words. 

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
general_summary = response

In [ ]:
prompt = f"""
Your task is to extract relevant information from  a movie review from a streaming service to give  feedback to the movie recommendation team.

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on   how the watcher felt about the movie sequence ending..

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
ending_summary = response

In [ ]:
prompt = f"""
Your task is to extract relevant information from  a movie review from a streaming service to give  feedback to the movie recommendation team.

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing how the watcher feels about the movie characters. 

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
character_summary = response

In [ ]:
prompt = f"""
Your task is to extract relevant information from  a movie review from a streaming service.


From the review below, delimited by triple backticks  extract the information relevant to ending sequence  Limit to 30 words. 

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
ending_extraction = response

In [ ]:
review_1 = movie_review 

# review for moview 2
review_2 = """
I was completely immersed in this cinematic gem! 
The storytelling is so engaging, and the characters feel like old friends. 
The mix of drama, humor, and suspense had me hooked throughout. 
Visually, it's a feast for the eyes with stunning effects, and the 
soundtrack added so much emotion. Sure, the ending was a bit expected, 
but the overall experience was so enjoyable that I highly recommend giving 
it a watch
"""

# review for movie 3
review_3 = """
What a journey! I felt like I was right there with 
the characters in this film. The storytelling is fantastic, and the 
characters are so well-developed that I genuinely cared about their fates. 
The blend of drama, humor, and suspense kept me entertained, 
and the visuals were breathtaking. Despite a few rushed scenes, 
I found myself thoroughly enjoying this movie. 
It's definitely worth your time.
"""

# review for movie 4
review_4 = """
I had a blast watching this film! The storytelling drew me in, 
and the characters were so relatable. The perfect mix of drama, humor, 
and suspense made it a rollercoaster of emotions. Visually, it's stunning, 
and the soundtrack added an extra layer of excitement. 
While the ending wasn't a huge surprise, 
the overall experience left me thoroughly entertained. 
If you're looking for a fun and engaging movie night, 
this one's a solid choice.
"""

reviews = [review_1, review_2, review_3, review_4]

In [ ]:
batch_summaries = []
for i in range(len(reviews)):
    prompt = f"""
    Your task is to extract relevant information from      a movie review from a streaming service.
    
    Summarize the review below, delimited by triple      backticks in at most 20 words. 

    Review: ```{reviews[i]}```
    """

    response = get_completion(prompt)
    batch_summaries.append(response)
    print(i, response, "\n")

In [ ]:
assert len(movie_review.split()) > 200
assert all(0 < len(t.split()) <= 30 for t in [general_summary, ending_summary, character_summary, ending_extraction])
assert "predictable" in ending_extraction.lower()
assert "character" in character_summary.lower()
assert len(reviews) == len(batch_summaries) == 4
assert all(0 < len(t.split()) <= 20 for t in batch_summaries)
assert len({t for t in batch_summaries}) >= 2
assert all(bad not in " ".join(batch_summaries).lower() for bad in ["shipping", "package delivery"])
print("Word counts:", json.dumps({"general": len(general_summary.split()), "ending": len(ending_summary.split()), "characters": len(character_summary.split()), "extraction": len(ending_extraction.split()), "four_reviews": [len(t.split()) for t in batch_summaries]}))
print("PASS: complete source review, all four batch inputs, measured 30/20-word limits and relevant extracted ending/character content.")

## Original archive appendix — not executed

Historical SDK fragments, examples and transcripts below are retained verbatim, not used as fresh test results.

### Original pre block 1

```text
pip install openai
```

### Original pre block 2

```text
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key  = os.getenv('OPENAI_API_KEY')
```

### Original pre block 3

```text
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content
```

### Original pre block 4

```text
movie_review = """
Watched this movie over the weekend, and it turned out to be a delightful cinematic experience. The storyline was engaging, weaving together various plot elements seamlessly. The characters were well-developed, and their interactions felt genuine. The film had a perfect blend of drama, humor, and suspense, keeping me hooked from start to finish.

The visuals were stunning, with impressive cinematography that added depth to the narrative. The special effects were top-notch, enhancing the overall viewing experience. The soundtrack was also noteworthy, complementing the on-screen action and evoking the right emotions at the right moments.

Despite these strengths, there were a few drawbacks. Some scenes felt a bit rushed, leaving me wanting more in terms of character development. Additionally, the ending seemed a tad predictable, lacking the surprise factor that could have elevated the film to greater heights.

On the positive side, the movie was a bit of a hidden gem, exceeding my expectations. The pacing was well-managed, and the runtime felt just right. It left me with a sense of satisfaction, much like finding a surprise extra feature on a DVD.

In conclusion, this film is worth a watch for its compelling story, impressive visuals, and engaging characters. While it may not be a groundbreaking masterpiece, it certainly provides an enjoyable and entertaining escape into its cinematic world.

"""
```

### Original pre block 5

```text
prompt = f"""
Your task is to generate a short summary of a movie \
review from a streaning service. 

Summarize the review below, delimited by triple 
backticks, in at most 30 words. 

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
```

### Original pre block 6

```text
prompt = f"""
Your task is to extract relevant information from \ 
a movie review from a streaming service to give \
feedback to the Shipping department.

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing on  \
how the watcher felt about the movie sequence ending..

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
```

### Original pre block 7

```text
prompt = f"""
Your task is to extract relevant information from \ 
a movie review from a streaming service to give \
feedback to the Shipping department.

Summarize the review below, delimited by triple 
backticks, in at most 30 words, and focusing how the watcher\
feels about the movie characters. 

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
```

### Original pre block 8

```text
prompt = f"""
Your task is to extract relevant information from \ 
a movie review from a streaming service.


From the review below, delimited by triple quotes \
extract the information relevant to ending sequence \ 
Limit to 30 words. 

Review: ```{movie_review}```
"""

response = get_completion(prompt)
print(response)
```

### Original pre block 9

```text
review_1 = movie_review 

# review for moview 2
review_2 = """
I was completely immersed in this cinematic gem! 
The storytelling is so engaging, and the characters feel like old friends. 
The mix of drama, humor, and suspense had me hooked throughout. 
Visually, it's a feast for the eyes with stunning effects, and the 
soundtrack added so much emotion. Sure, the ending was a bit expected, 
but the overall experience was so enjoyable that I highly recommend giving 
it a watch
"""

# review for movie 3
review_3 = """
What a journey! I felt like I was right there with 
the characters in this film. The storytelling is fantastic, and the 
characters are so well-developed that I genuinely cared about their fates. 
The blend of drama, humor, and suspense kept me entertained, 
and the visuals were breathtaking. Despite a few rushed scenes, 
I found myself thoroughly enjoying this movie. 
It's definitely worth your time.
"""

# review for movie 4
review_4 = """
I had a blast watching this film! The storytelling drew me in, 
and the characters were so relatable. The perfect mix of drama, humor, 
and suspense made it a rollercoaster of emotions. Visually, it's stunning, 
and the soundtrack added an extra layer of excitement. 
While the ending wasn't a huge surprise, 
the overall experience left me thoroughly entertained. 
If you're looking for a fun and engaging movie night, 
this one's a solid choice.
"""

reviews = [review_1, review_2, review_3, review_4]
```

### Original pre block 10

```text
for i in range(len(reviews)):
    prompt = f"""
    Your task is to extract relevant information from \ 
    a movie review from a streaming service.
    
    Summarize the review below, delimited by triple \
    backticks in at most 20 words. 

    Review: ```{reviews[i]}```
    """

    response = get_completion(prompt)
    print(i, response, "\n")
```

### Original transcript 1

```text
This movie offers a delightful cinematic experience with engaging storytelling, well-developed characters, and a perfect blend of drama, humor, and suspense. The visuals and special effects are stunning, and the soundtrack complements the action. However, some scenes feel rushed and the ending is predictable. Overall, it is an enjoyable and entertaining film worth watching.
```

### Original transcript 2

```text
The watcher enjoyed the movie’s storyline, characters, visuals, and soundtrack. However, they felt that some scenes were rushed and the ending was predictable, lacking surprise. Overall, the movie was enjoyable and worth watching.
```

### Original transcript 3

```text
The watcher feels that the movie characters were well-developed and their interactions felt genuine, making the film an enjoyable and entertaining escape.
```

### Original transcript 4

```text
The ending seemed predictable, lacking surprise factor.
```

### Original transcript 5

```text
0 A delightful cinematic experience with engaging storyline, stunning visuals, and impressive soundtrack, despite rushed scenes and predictable ending.
```

### Original transcript 6

```text
1 A highly enjoyable cinematic gem with engaging storytelling, stunning visuals, and a captivating mix of drama, humor, and suspense.
```

### Original transcript 7

```text
2 A captivating film with well-developed characters, a great blend of drama, humor, and suspense, and breathtaking visuals. Worth watching.
```

### Original transcript 8

```text
3 A highly entertaining film with relatable characters, a rollercoaster of emotions, stunning visuals, and an exciting soundtrack.
```